In [0]:
# ============================================================
# 02_clean_documents
# ============================================================
#
# Purpose:
#   Clean text extracted from the GEP PDFs while preserving:
#     - document/page identity
#     - parsing quality status
#     - headings
#     - numbers
#     - economic terminology
#     - paragraph boundaries
#     - citation/page information
#
# Input:
#   worldbank_ai.silver.gep_parsed_pages
#
# Output:
#   worldbank_ai.silver.gep_clean_pages
#
# IMPORTANT:
#   This notebook does NOT:
#     - chunk documents
#     - summarize text
#     - use an LLM
#     - remove NO_TEXT pages
#     - remove VISUAL_HEAVY pages
# ============================================================

CATALOG = "worldbank_ai"
SCHEMA = "silver"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.gep_parsed_pages"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.gep_clean_pages"

EXPECTED_DOCUMENTS = 5
EXPECTED_PAGES = 1098

print(f"Source table: {SOURCE_TABLE}")
print(f"Target table: {TARGET_TABLE}")

In [0]:
# ============================================================
# Imports
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
# ============================================================
# Load parsed GEP pages
# ============================================================

parsed_df = spark.table(SOURCE_TABLE)

parsed_count = parsed_df.count()

document_count = (
    parsed_df
    .select("document_id")
    .distinct()
    .count()
)

print(f"Parsed pages:     {parsed_count:,}")
print(f"GEP documents:    {document_count}")

In [0]:
# ============================================================
# Validate source data before cleaning
# ============================================================

if parsed_count != EXPECTED_PAGES:
    raise RuntimeError(
        f"Expected {EXPECTED_PAGES:,} parsed pages, "
        f"but found {parsed_count:,}."
    )

if document_count != EXPECTED_DOCUMENTS:
    raise RuntimeError(
        f"Expected {EXPECTED_DOCUMENTS} documents, "
        f"but found {document_count}."
    )


# Validate unique page IDs.
duplicate_page_ids = (
    parsed_df
    .groupBy("page_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate page IDs: {duplicate_page_ids}")

if duplicate_page_ids != 0:
    raise RuntimeError(
        "Duplicate page IDs detected."
    )

print("Source validation passed.")

In [0]:
# ============================================================
# Validate parsing-status column
# ============================================================
#
# We created this in the parsing notebook.
#
# We keep this metadata through the entire unstructured
# pipeline because it will later help identify:
#
#   TEXT_OK
#   LOW_TEXT
#   VISUAL_HEAVY
#   NO_TEXT
# ============================================================

if "parsing_status" not in parsed_df.columns:
    raise RuntimeError(
        "Column 'parsing_status' was not found. "
        "Rerun 01_parse_documents after adding the "
        "parsing-status logic."
    )


display(
    parsed_df
    .groupBy(
        "report_year",
        "parsing_status"
    )
    .count()
    .orderBy(
        "report_year",
        "parsing_status"
    )
)

In [0]:
# ============================================================
# Start from the original extracted text
# ============================================================
#
# raw_text remains unchanged.
#
# clean_text becomes the cleaned version.
#
# Keeping both gives us:
#   - traceability
#   - debugging
#   - before/after comparison
#   - ability to improve cleaning later
# ============================================================

clean_df = (
    parsed_df

    .withColumn(
        "clean_text",
        F.col("raw_text")
    )
)

In [0]:
# ============================================================
# Normalize control characters and whitespace
# ============================================================

clean_df = (
    clean_df

    # --------------------------------------------------------
    # Remove NULL control characters.
    # --------------------------------------------------------
    .withColumn(
        "clean_text",
        F.regexp_replace(
            F.col("clean_text"),
            "\u0000",
            ""
        )
    )

    # --------------------------------------------------------
    # Convert non-breaking spaces to regular spaces.
    # --------------------------------------------------------
    .withColumn(
        "clean_text",
        F.regexp_replace(
            F.col("clean_text"),
            "\u00A0",
            " "
        )
    )

    # --------------------------------------------------------
    # Normalize Windows/Mac line endings.
    #
    # \r\n -> \n
    # \r   -> \n
    # --------------------------------------------------------
    .withColumn(
        "clean_text",
        F.regexp_replace(
            F.col("clean_text"),
            r"\r\n?",
            "\n"
        )
    )

    # --------------------------------------------------------
    # Normalize tabs to spaces.
    # --------------------------------------------------------
    .withColumn(
        "clean_text",
        F.regexp_replace(
            F.col("clean_text"),
            r"\t+",
            " "
        )
    )
)

In [0]:
# ============================================================
# Repair words broken by PDF line wrapping
# ============================================================
#
# Example:
#
#     eco-
#     nomic
#
# becomes:
#
#     economic
#
# We only apply this when:
#   - character before "-" is alphabetic
#   - next line begins with lowercase alphabetic character
#
# This is intentionally conservative.
# ============================================================

clean_df = (
    clean_df

    .withColumn(
        "clean_text",

        F.regexp_replace(
            F.col("clean_text"),
            r"([A-Za-z])-\s*\n\s*([a-z])",
            "$1$2"
        )
    )
)

In [0]:
# ============================================================
# Normalize horizontal whitespace
# ============================================================
#
# Collapse repeated spaces/tabs while keeping newline
# boundaries intact.
# ============================================================

clean_df = (
    clean_df

    .withColumn(
        "clean_text",

        F.regexp_replace(
            F.col("clean_text"),
            r"[ \t]+",
            " "
        )
    )
)

In [0]:
# ============================================================
# Remove obvious repeated report header/footer artifacts
# ============================================================
#
# Only remove lines that consist entirely of the report title.
#
# We do NOT remove occurrences of "Global Economic Prospects"
# when they appear as part of actual sentences.
# ============================================================

clean_df = (
    clean_df

    .withColumn(
        "clean_text",

        F.regexp_replace(
            F.col("clean_text"),
            r"(?im)^\s*GLOBAL ECONOMIC PROSPECTS\s*$",
            ""
        )
    )
)

In [0]:
# ============================================================
# Normalize excessive blank lines
# ============================================================
#
# Preserve paragraph separation but collapse large runs of
# blank lines to a maximum of one blank line.
# ============================================================

clean_df = (
    clean_df

    .withColumn(
        "clean_text",

        F.regexp_replace(
            F.col("clean_text"),
            r"\n\s*\n\s*\n+",
            "\n\n"
        )
    )

    .withColumn(
        "clean_text",
        F.trim(
            F.col("clean_text")
        )
    )
)

In [0]:
# ============================================================
# Add cleaning diagnostics
# ============================================================

clean_df = (
    clean_df

    # Number of characters after cleaning.
    .withColumn(
        "clean_character_count",
        F.length(
            F.coalesce(
                F.col("clean_text"),
                F.lit("")
            )
        )
    )

    # Number of characters removed.
    .withColumn(
        "characters_removed",

        F.col("character_count")
        - F.col("clean_character_count")
    )

    # Percentage of original characters retained.
    .withColumn(
        "character_retention_pct",

        F.when(
            F.col("character_count") > 0,

            F.round(
                (
                    F.col("clean_character_count")
                    / F.col("character_count")
                ) * 100,
                2
            )
        )
    )

    # Flag pages with no text after cleaning.
    .withColumn(
        "is_empty_after_cleaning",

        F.col("clean_character_count") == 0
    )

    # Processing timestamp.
    .withColumn(
        "cleaned_at",
        F.current_timestamp()
    )
)

In [0]:
# ============================================================
# Validate page preservation
# ============================================================

clean_count = clean_df.count()

print(f"Parsed pages: {parsed_count:,}")
print(f"Clean pages:  {clean_count:,}")


if clean_count != parsed_count:
    raise RuntimeError(
        "Cleaning changed the number of page records."
    )

print("Page-count validation passed.")

In [0]:
# ============================================================
# Overall cleaning statistics
# ============================================================

cleaning_stats = (
    clean_df

    .agg(

        F.count("*").alias(
            "total_pages"
        ),

        F.sum(
            F.when(
                F.col("is_empty_after_cleaning"),
                1
            ).otherwise(0)
        ).alias(
            "empty_pages"
        ),

        F.sum(
            "character_count"
        ).alias(
            "original_characters"
        ),

        F.sum(
            "clean_character_count"
        ).alias(
            "clean_characters"
        ),

        F.sum(
            "characters_removed"
        ).alias(
            "characters_removed"
        ),

        F.round(
            F.avg(
                "character_retention_pct"
            ),
            2
        ).alias(
            "avg_character_retention_pct"
        )
    )
)

display(cleaning_stats)

In [0]:
# ============================================================
# Cleaning statistics by GEP edition
# ============================================================

cleaning_by_report_df = (
    clean_df

    .groupBy(
        "report_year"
    )

    .agg(

        F.count("*").alias(
            "pages"
        ),

        F.sum(
            "character_count"
        ).alias(
            "original_characters"
        ),

        F.sum(
            "clean_character_count"
        ).alias(
            "clean_characters"
        ),

        F.sum(
            "characters_removed"
        ).alias(
            "characters_removed"
        ),

        F.round(
            (
                F.sum("clean_character_count")
                / F.sum("character_count")
            ) * 100,
            2
        ).alias(
            "character_retention_pct"
        )
    )

    .orderBy(
        "report_year"
    )
)

display(cleaning_by_report_df)

In [0]:
# ============================================================
# Detect suspicious text loss
# ============================================================
#
# A page losing a very large percentage of its extracted text
# deserves inspection.
#
# This does NOT automatically mean there is an error.
# ============================================================

suspicious_cleaning_df = (
    clean_df

    .filter(
        (F.col("character_count") >= 500)
        &
        (
            F.col("character_retention_pct") < 80
        )
    )

    .select(
        "document_id",
        "report_year",
        "page_number",
        "parsing_status",
        "character_count",
        "clean_character_count",
        "character_retention_pct",
        "raw_text",
        "clean_text"
    )

    .orderBy(
        "report_year",
        "page_number"
    )
)

suspicious_count = (
    suspicious_cleaning_df.count()
)

print(
    f"Pages with <80% character retention: "
    f"{suspicious_count}"
)

display(
    suspicious_cleaning_df
)

In [0]:
# ============================================================
# Inspect empty pages after cleaning
# ============================================================

empty_pages_df = (
    clean_df

    .filter(
        F.col("is_empty_after_cleaning")
    )

    .select(
        "document_id",
        "report_year",
        "page_number",
        "parsing_status",
        "character_count",
        "image_count"
    )

    .orderBy(
        "report_year",
        "page_number"
    )
)

empty_page_count = (
    empty_pages_df.count()
)

print(
    f"Empty pages after cleaning: "
    f"{empty_page_count}"
)

display(
    empty_pages_df
)

In [0]:
# ============================================================
# Validate parsing-status preservation
# ============================================================

display(
    clean_df

    .groupBy(
        "report_year",
        "parsing_status"
    )

    .count()

    .orderBy(
        "report_year",
        "parsing_status"
    )
)

In [0]:
# ============================================================
# Compare raw and cleaned text
# ============================================================

display(
    clean_df

    .filter(
        F.col("character_count") > 1000
    )

    .select(
        "report_year",
        "page_number",
        "parsing_status",
        "character_count",
        "clean_character_count",
        "character_retention_pct",
        "raw_text",
        "clean_text"
    )

    .orderBy(
        "report_year",
        "page_number"
    )

    .limit(20)
)

In [0]:
# ============================================================
# Verify important structural headings survived cleaning
# ============================================================

heading_check_df = (
    clean_df

    .filter(
        F.lower(
            F.coalesce(
                F.col("clean_text"),
                F.lit("")
            )
        ).rlike(
            r"(^|\n)(outlook|risks|recent developments|policy challenges)(\n|$)"
        )
    )

    .select(
        "report_year",
        "page_number",
        "clean_text"
    )

    .orderBy(
        "report_year",
        "page_number"
    )
)

heading_page_count = (
    heading_check_df.count()
)

print(
    f"Pages containing structural headings: "
    f"{heading_page_count}"
)

display(
    heading_check_df.limit(30)
)

In [0]:
# ============================================================
# Regression test: preserve numeric table values
# ============================================================
#
# During cleaning validation we discovered that removing
# standalone numeric lines corrupted table data.
#
# Page 170 of the 2022 report contains commodity-cycle data.
# These numeric values must survive cleaning.
#
# Keeping this test protects us from accidentally introducing
# the same cleaning bug later.
# ============================================================

table_test_row = (
    clean_df

    .filter(
        (F.col("report_year") == 2022)
        &
        (F.col("page_number") == 170)
    )

    .select(
        "clean_text"
    )

    .first()
)


if table_test_row is None:
    raise RuntimeError(
        "Regression-test page 170 was not found."
    )


table_text = table_test_row["clean_text"]


# Values known to appear in the Banana, Europe row.
required_values = [
    "Banana, Europe*",
    "3",
    "20",
    "53",
    "93",
    "-55",
    "4.5",
    "-1.0"
]


missing_values = [
    value
    for value in required_values
    if value not in table_text
]


if missing_values:

    raise RuntimeError(
        "Table-value preservation regression failed. "
        f"Missing values: {missing_values}"
    )


print(
    "Table-value preservation regression test passed."
)

In [0]:
# ============================================================
# Select final Silver clean-page schema
# ============================================================

final_clean_df = (
    clean_df

    .select(

        # ----------------------------------------------------
        # Document identity
        # ----------------------------------------------------

        "document_id",
        "page_id",

        "report_year",

        "filename",
        "file_path",

        # ----------------------------------------------------
        # Page identity / citation
        # ----------------------------------------------------

        "page_number",

        # ----------------------------------------------------
        # Original parsed content
        # ----------------------------------------------------

        "raw_text",

        # ----------------------------------------------------
        # Cleaned content
        # ----------------------------------------------------

        "clean_text",

        # ----------------------------------------------------
        # Parsing diagnostics
        # ----------------------------------------------------

        "parsing_status",

        "text_block_count",
        "image_count",

        "page_width",
        "page_height",

        # ----------------------------------------------------
        # Cleaning diagnostics
        # ----------------------------------------------------

        "character_count",
        "clean_character_count",
        "characters_removed",
        "character_retention_pct",

        "is_empty_after_cleaning",

        # ----------------------------------------------------
        # Lineage
        # ----------------------------------------------------

        "parsed_at",
        "cleaned_at"
    )
)

In [0]:
# ============================================================
# Final validation before persistence
# ============================================================

final_count = (
    final_clean_df.count()
)

unique_pages = (
    final_clean_df
    .select("page_id")
    .distinct()
    .count()
)

documents = (
    final_clean_df
    .select("document_id")
    .distinct()
    .count()
)

report_years = [
    row["report_year"]
    for row in (
        final_clean_df
        .select("report_year")
        .distinct()
        .orderBy("report_year")
        .collect()
    )
]


print("=" * 60)

print("FINAL CLEANING VALIDATION")

print("=" * 60)

print(
    f"Input pages:        {parsed_count:,}"
)

print(
    f"Output pages:       {final_count:,}"
)

print(
    f"Unique page IDs:    {unique_pages:,}"
)

print(
    f"Documents:          {documents}"
)

print(
    f"Report years:       {report_years}"
)

print(
    f"Empty clean pages:  {empty_page_count:,}"
)


if final_count != parsed_count:
    raise RuntimeError(
        "Input/output page counts do not match."
    )

if unique_pages != final_count:
    raise RuntimeError(
        "Duplicate page IDs detected."
    )

if documents != 5:
    raise RuntimeError(
        "Expected exactly five GEP documents."
    )

if report_years != [
    2022,
    2023,
    2024,
    2025,
    2026
]:
    raise RuntimeError(
        "Unexpected report-year coverage."
    )


print("=" * 60)
print("Cleaning validation passed.")
print("=" * 60)

In [0]:
# ============================================================
# Persist cleaned GEP pages
# ============================================================

(
    final_clean_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Saved cleaned GEP pages to:\n"
    f"{TARGET_TABLE}"
)

In [0]:
# ============================================================
# Read-back validation
# ============================================================
#
# Validate what was actually persisted, not just the in-memory
# DataFrame.
# ============================================================

saved_df = spark.table(
    TARGET_TABLE
)

saved_count = (
    saved_df.count()
)

saved_unique_pages = (
    saved_df
    .select("page_id")
    .distinct()
    .count()
)

saved_documents = (
    saved_df
    .select("document_id")
    .distinct()
    .count()
)


print("=" * 60)

print("PERSISTED TABLE VALIDATION")

print("=" * 60)

print(
    f"Saved pages:        {saved_count:,}"
)

print(
    f"Unique page IDs:    {saved_unique_pages:,}"
)

print(
    f"Documents:          {saved_documents}"
)


if saved_count != final_count:
    raise RuntimeError(
        "Persisted row count does not match expected count."
    )

if saved_unique_pages != saved_count:
    raise RuntimeError(
        "Duplicate page IDs exist in persisted table."
    )

if saved_documents != 5:
    raise RuntimeError(
        "Persisted table does not contain five documents."
    )


print("=" * 60)
print("02_clean_documents completed successfully.")
print("=" * 60)